# Sélection du nombre optimal de clusters & Mesure de qualité

Ce notebook illustre les méthodes de sélection du K optimal et d'évaluation de la qualité
d'un clustering, en utilisant uniquement les fonctions from-scratch définies dans `kmeans.py`.

## Plan
1. Chargement / génération des données
2. Recherche du K optimal avec `find_optimal_k()`
3. Visualisation de la courbe du coude (inertie)
4. Visualisation du score de silhouette
5. Visualisation de l'indice de Davies-Bouldin
6. Visualisation de l'indice de Calinski-Harabasz
7. Visualisation de la Gap Statistic
8. Synthèse : quel K choisir ?
9. Validation : clustering final

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from kmeans import KMeans, find_optimal_k, silhouette_score, davies_bouldin_score, calinski_harabasz_score, gap_statistic, _build_demo_data

%matplotlib inline

## 1. Données

On utilise le jeu de données synthétiques à 3 clusters fourni par `_build_demo_data()`,
qui génère 300 points en 2D répartis en 3 groupes gaussiens bien séparés.

In [ ]:
# Génération des données
data = _build_demo_data(seed=7)
print(f"Forme des données : {data.shape}")
print(f"Min / Max : ({data.min(axis=0)}) / ({data.max(axis=0)})")

# Aperçu visuel
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(data[:, 0], data[:, 1], c="steelblue", alpha=0.6, edgecolors="k", s=30)
ax.set_title("Données brutes (3 clusters gaussiens)")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 2. Recherche automatique du K optimal

La fonction `find_optimal_k()` teste K de 2 à `max_k` (ici 10), entraîne un `KMeans`
pour chaque valeur, et calcule les 5 métriques de qualité : inertie, silhouette,
Davies-Bouldin, Calinski-Harabasz et Gap Statistic.

In [ ]:
# Lancement de la recherche
results = find_optimal_k(data, max_k=10, random_state=7)

# Affichage du tableau récapitulatif
header = f"{'K':<5} {'Inertie':>12} {'Silhouette':>12} {'Davies-Bouldin':>16} {'Calinski-Harabasz':>20} {'Gap':>10}"
print(header)
print("-" * 80)
for k, inert, sil, db, ch, gap in zip(
    results["ks"],
    results["inertias"],
    results["silhouette_scores"],
    results["davies_bouldin_scores"],
    results["calinski_harabasz_scores"],
    results["gap_scores"],
):
    print(f"{k:<5} {inert:>12.2f} {sil:>12.4f} {db:>16.4f} {ch:>20.2f} {gap:>10.4f}")

print("\n--- K optimal détecté ---")
print(f"Méthode du coude          → K = {results['best_k_elbow']}")
print(f"Score de silhouette       → K = {results['best_k_silhouette']}")
print(f"Indice de Davies-Bouldin  → K = {results['best_k_davies_bouldin']}")
print(f"Indice de Calinski-Harabasz → K = {results['best_k_calinski_harabasz']}")
print(f"Gap Statistic             → K = {results['best_k_gap']}")

## 3. Méthode du coude (Elbow method)

L'inertie (somme des distances au carré au centroïde) diminue quand K augmente.
Le coude est le point où la décroissance ralentit fortement — c'est le K optimal.

On inclut K=1 (un seul cluster contenant tous les points) pour que la courbe ait du sens.

In [ ]:
# Calcul de l'inertie pour K=1
global_center = data.mean(axis=0)
inertia_k1 = float(np.sum((data - global_center) ** 2))

all_ks = [1] + results["ks"]
all_inertias = [inertia_k1] + results["inertias"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(all_ks, all_inertias, marker="o", linewidth=2, markersize=8, color="steelblue")

elbow_k = results["best_k_elbow"]
elbow_idx = elbow_k - 1
ax.scatter([elbow_k], [all_inertias[elbow_idx]], color="crimson", s=200, zorder=5,
           edgecolors="black", linewidth=1.5, label=f"Coude détecté (K={elbow_k})")

ax.set_xlabel("Nombre de clusters K")
ax.set_ylabel("Inertie")
ax.set_title("Méthode du coude — Inertie en fonction de K")
ax.set_xticks(all_ks)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Score de silhouette

Le score de silhouette mesure à quel point chaque point est bien classé :
- Proche de **+1** : le point est bien dans son cluster et loin des autres
- Proche de **0**  : clusters qui se chevauchent
- Négatif         : point probablement mal classé

**Règle : plus le score est élevé, meilleur est le clustering.**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(results["ks"], results["silhouette_scores"], marker="s", linewidth=2,
       markersize=8, color="darkgreen")

best_k = results["best_k_silhouette"]
best_idx = results["ks"].index(best_k)
ax.scatter([best_k], [results["silhouette_scores"][best_idx]], color="crimson", s=200,
           zorder=5, edgecolors="black", linewidth=1.5, label=f"Meilleur K={best_k}")

ax.set_xlabel("Nombre de clusters K")
ax.set_ylabel("Score de silhouette")
ax.set_title("Score de silhouette en fonction de K")
ax.set_xticks(results["ks"])
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Indice de Davies-Bouldin

L'indice de Davies-Bouldin compare la dispersion interne des clusters à la distance
entre leurs centroïdes.

**Règle : plus l'indice est petit, meilleur est le clustering** (clusters compacts et bien séparés).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(results["ks"], results["davies_bouldin_scores"], marker="D", linewidth=2,
       markersize=8, color="darkorange")

best_k = results["best_k_davies_bouldin"]
best_idx = results["ks"].index(best_k)
ax.scatter([best_k], [results["davies_bouldin_scores"][best_idx]], color="crimson", s=200,
           zorder=5, edgecolors="black", linewidth=1.5, label=f"Meilleur K={best_k}")

ax.set_xlabel("Nombre de clusters K")
ax.set_ylabel("Indice de Davies-Bouldin")
ax.set_title("Indice de Davies-Bouldin en fonction de K")
ax.set_xticks(results["ks"])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Indice de Calinski-Harabasz

L'indice de Calinski-Harabasz (Variance Ratio Criterion) compare la dispersion inter-cluster
à la dispersion intra-cluster.

**Règle : plus l'indice est grand, meilleur est le clustering** (clusters denses et bien séparés).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(results["ks"], results["calinski_harabasz_scores"], marker="^", linewidth=2,
       markersize=8, color="mediumpurple")

best_k = results["best_k_calinski_harabasz"]
best_idx = results["ks"].index(best_k)
ax.scatter([best_k], [results["calinski_harabasz_scores"][best_idx]], color="crimson", s=200,
           zorder=5, edgecolors="black", linewidth=1.5, label=f"Meilleur K={best_k}")

ax.set_xlabel("Nombre de clusters K")
ax.set_ylabel("Indice de Calinski-Harabasz")
ax.set_title("Indice de Calinski-Harabasz en fonction de K")
ax.set_xticks(results["ks"])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Gap Statistic (Statistique d'écart)

La Gap Statistic (Tibshirani, Walther & Hastie, 2001) compare l'inertie observée sur
les données réelles à l'inertie moyenne obtenue sur des données de référence uniformément
distribuées (sans structure de clusters).

$$\text{Gap}(K) = \frac{1}{B} \sum_{b=1}^{B} \log(W^*_{k,b}) - \log(W_k)$$

où $W_k$ est l'inertie sur les données réelles et $W^*_{k,b}$ l'inertie sur le $b$-ième
jeu de référence (uniforme dans le bounding-box des données).

**Règle : plus le Gap est grand, plus K est pertinent** (l'inertie observée est bien plus petite que l'inertie attendue sous distribution uniforme → il existe une vraie structure de clusters).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(results["ks"], results["gap_scores"], marker="v", linewidth=2,
       markersize=8, color="teal")

best_k = results["best_k_gap"]
best_idx = results["ks"].index(best_k)
ax.scatter([best_k], [results["gap_scores"][best_idx]], color="crimson", s=200,
           zorder=5, edgecolors="black", linewidth=1.5, label=f"Meilleur K={best_k}")

ax.set_xlabel("Nombre de clusters K")
ax.set_ylabel("Gap(K)")
ax.set_title("Gap Statistic en fonction de K")
ax.set_xticks(results["ks"])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Synthèse : quel K choisir ?

On applique la méthodologie en 5 étapes :

1. **Tester** plusieurs valeurs de K (2 à 10)
2. **Regarder la méthode du coude** pour identifier une zone plausible
3. **Vérifier avec la silhouette** (prendre un K avec un score élevé)
4. **Confirmer avec la Gap Statistic**, Davies-Bouldin et Calinski-Harabasz
5. **Garder un K interprétable** métier

Ici les 5 méthodes s'accordent sur K=3, ce qui correspond exactement aux 3 clusters
générés dans les données synthétiques.

In [ ]:
# Synthèse visuelle : les 5 métriques dans une grille 3×2
fig, axes = plt.subplots(3, 2, figsize=(14, 16))

# Inertie (coude)
ax = axes[0, 0]
ax.plot(all_ks, all_inertias, marker="o", color="steelblue", linewidth=2, markersize=8)
ax.scatter([results["best_k_elbow"]], [all_inertias[results["best_k_elbow"] - 1]],
          color="crimson", s=150, zorder=5, edgecolors="black")
ax.set_xlabel("K"); ax.set_ylabel("Inertie"); ax.set_title("Méthode du coude")
ax.set_xticks(all_ks); ax.grid(True, alpha=0.3)

# Silhouette
ax = axes[0, 1]
ax.plot(results["ks"], results["silhouette_scores"], marker="s", color="darkgreen", linewidth=2, markersize=8)
ax.scatter([results["best_k_silhouette"]],
          [results["silhouette_scores"][results["ks"].index(results["best_k_silhouette"])]],
          color="crimson", s=150, zorder=5, edgecolors="black")
ax.set_xlabel("K"); ax.set_ylabel("Score"); ax.set_title("Silhouette (± proche de 1 = meilleur)")
ax.set_xticks(results["ks"]); ax.grid(True, alpha=0.3)
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)

# Davies-Bouldin
ax = axes[1, 0]
ax.plot(results["ks"], results["davies_bouldin_scores"], marker="D", color="darkorange", linewidth=2, markersize=8)
ax.scatter([results["best_k_davies_bouldin"]],
          [results["davies_bouldin_scores"][results["ks"].index(results["best_k_davies_bouldin"])]],
          color="crimson", s=150, zorder=5, edgecolors="black")
ax.set_xlabel("K"); ax.set_ylabel("Score"); ax.set_title("Davies-Bouldin (± petit = meilleur)")
ax.set_xticks(results["ks"]); ax.grid(True, alpha=0.3)

# Calinski-Harabasz
ax = axes[1, 1]
ax.plot(results["ks"], results["calinski_harabasz_scores"], marker="^", color="mediumpurple", linewidth=2, markersize=8)
ax.scatter([results["best_k_calinski_harabasz"]],
          [results["calinski_harabasz_scores"][results["ks"].index(results["best_k_calinski_harabasz"])]],
          color="crimson", s=150, zorder=5, edgecolors="black")
ax.set_xlabel("K"); ax.set_ylabel("Score"); ax.set_title("Calinski-Harabasz (± grand = meilleur)")
ax.set_xticks(results["ks"]); ax.grid(True, alpha=0.3)

# Gap Statistic
ax = axes[2, 0]
ax.plot(results["ks"], results["gap_scores"], marker="v", color="teal", linewidth=2, markersize=8)
ax.scatter([results["best_k_gap"]],
          [results["gap_scores"][results["ks"].index(results["best_k_gap"])]],
          color="crimson", s=150, zorder=5, edgecolors="black")
ax.set_xlabel("K"); ax.set_ylabel("Gap(K)"); ax.set_title("Gap Statistic (± grand = meilleur)")
ax.set_xticks(results["ks"]); ax.grid(True, alpha=0.3)

# Axe vide (la 6e cellule est laissée libre)
axes[2, 1].axis("off")

fig.suptitle("Synthèse des 5 métriques de sélection du K optimal", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# Résumé textuel
print("Résumé des K optimaux :")
print(f"  Coude            → K = {results['best_k_elbow']}")
print(f"  Silhouette       → K = {results['best_k_silhouette']}  (score = {results['silhouette_scores'][results['ks'].index(results['best_k_silhouette'])]:.4f})")
print(f"  Davies-Bouldin   → K = {results['best_k_davies_bouldin']}  (score = {results['davies_bouldin_scores'][results['ks'].index(results['best_k_davies_bouldin'])]:.4f})")
print(f"  Calinski-Harabasz → K = {results['best_k_calinski_harabasz']}  (score = {results['calinski_harabasz_scores'][results['ks'].index(results['best_k_calinski_harabasz'])]:.2f})")
print(f"  Gap Statistic    → K = {results['best_k_gap']}  (gap = {results['gap_scores'][results['ks'].index(results['best_k_gap'])]:.4f})")
print(f"\n→ Consensus : K = {results['best_k_silhouette']}")

## 9. Validation : clustering final avec K=3

On entraîne un KMeans avec le K optimal trouvé (3) et on visualise les clusters obtenus.

In [ ]:
# Entraînement final
final_model = KMeans(n_clusters=3, random_state=7, init="k-means++")
final_model.fit(data)

print(f"Inertie finale    : {final_model.inertia_:.2f}")
print(f"Itérations        : {final_model.n_iter_}")
print(f"Silhouette        : {silhouette_score(data, final_model.labels_):.4f}")
print(f"Davies-Bouldin    : {davies_bouldin_score(data, final_model.labels_):.4f}")
print(f"Calinski-Harabasz : {calinski_harabasz_score(data, final_model.labels_):.2f}")
print(f"Gap Statistic     : {gap_statistic(data, k=3, random_state=7):.4f}")

# Visualisation des clusters
fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#e74c3c", "#3498db", "#2ecc71"]

for cluster_idx in range(3):
    mask = final_model.labels_ == cluster_idx
    ax.scatter(data[mask, 0], data[mask, 1], c=colors[cluster_idx],
              alpha=0.7, edgecolors="k", s=40, label=f"Cluster {cluster_idx}")

# Centroïdes
ax.scatter(final_model.cluster_centers_[:, 0], final_model.cluster_centers_[:, 1],
          c="gold", marker="X", s=250, edgecolors="black", linewidth=2,
          label="Centroïdes", zorder=10)

ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_title(f"Clustering final — K={3} clusters")
ax.set_aspect("equal")
ax.legend()
plt.tight_layout()
plt.show()

---

### Résumé des métriques implémentées

| Métrique | Formule / Principe | Règle |
|---|---|---|
| **Inertie (Elbow)** | Somme des distances² au centroïde le plus proche | Chercher le « coude » |
| **Silhouette** | $s(i) = (b(i) - a(i)) / \max(a(i), b(i))$ | + proche de 1 = meilleur |
| **Davies-Bouldin** | $R_k = \max_{j \neq k} (s_k + s_j) / d(c_k, c_j)$ | + petit = meilleur |
| **Calinski-Harabasz** | $CH = (\mathrm{tr}(B_k) / \mathrm{tr}(W_k)) \cdot ((n-k)/(k-1))$ | + grand = meilleur |
| **Gap Statistic** | $\text{Gap}(K) = \frac{1}{B} \sum \log(W^*_k) - \log(W_k)$ | + grand = meilleur |

Toutes ces fonctions sont implémentées **from scratch** dans `kmeans.py` en utilisant uniquement NumPy.